### Dataset Inspection

When facing an ML problem, the very first thing you shall **always** do before starting your training endeavours is to inspect the dataset. In other words, look at the data, make some short analysis (and meta-analysis) to get a feeling about the type of data you are dealing with. We will demonstrate some basic yet useful techniques in this notebook.

Throughout this notebook, we will be using the famous MNIST (Modified National Institute of Standards and Technology) dataset.

#### MNIST (Dataset Definition)

Before loading any data, you shall learn about the dataset itself, about its definition and structure. In our case, we will refer to the official documentation that you can find [here](http://yann.lecun.com/exdb/mnist/).

There you learn that the dataset consists of 70 000 samples (60k for train and 10k for test). Each sample is a 28x28 monochrome image containing a handwritten digit. It follows that there are 10 different classes (one per each digit). In the provided link, you can also read about how the dataset has been built.

#### Dataset Visualisation

The next step is usually to look at the data themselves. In the beginning, load a subset of random data and visualize them. That way, you will get an impression of what type of data (in this case, what type of handwriting) we are dealing with.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

import keras
import numpy as np

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

# Set the seed for reproducibility (python, numpy and the backend)
keras.utils.set_random_seed(1234)

MNIST is one of the Keras built-in datasets and it can be easily loaded.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Dataset params
num_classes = 10
size = x_train.shape[1]

print('Train set:   ', len(y_train), 'samples')
print('Test set:    ', len(y_test), 'samples')
print('Sample dims: ', x_train.shape)

Besides the number of samples, always check the **data type** and the **range of values** of the data. Here the images are `uint8`, i.e., the pixel values live in the range [0, 255], while the labels are the class IDs 0-9. Keep this in mind, we will come back to it in the next notebook (dynamic range).

In [ ]:
print('Image dtype: ', x_train.dtype)
print('Pixel range: ', x_train.min(), '-', x_train.max())
print('Label dtype: ', y_train.dtype)
print('Label values:', np.unique(y_train))

Let's now visualise random subsets of data.

In [ ]:
for ii in range(15):
    idx = np.random.randint(0, len(y_train))
    plt.subplot(3,5,ii+1), plt.imshow(x_train[idx, ...], cmap='gray'), plt.title(y_train[idx])

#### Class Histograms

In order to correctly design your learning (training) strategies, it is important to analyse how well the different classes are represented in the dataset. In other words, we are trying to answer the following questions:
* Do we have enough samples for each class?
* Are there classes that are under-represented?
* Are there classes that are over-represented?
* Is the class distribution reflecting the real world?

In order to answer these questions, a very powerful tool is to plot the class histogram.

In [ ]:
counts = np.bincount(y_train)

plt.bar(range(num_classes), counts), plt.grid(True)
plt.xticks(range(num_classes))
plt.xlabel('Class ID'), plt.ylabel('counts')

Even though the class distribution is not totally homogeneous, we can conclude that no class is significantly over/under-represented. In relative terms, the histogram looks as shown below. A perfectly balanced dataset would have 10% of samples of each class (the dashed line).

In [ ]:
plt.bar(range(num_classes), counts/np.sum(counts)), plt.grid(True)
plt.axhline(1/num_classes, color='r', linestyle='--')
plt.xticks(range(num_classes))
plt.xlabel('Class ID'), plt.ylabel('fraction of samples')

Why do we care about the balance? Imagine a dataset of 1000 frames with 990 cars and 10 motorcycles. A "classifier" that says *car* for everything has an accuracy of 99 %, and yet it is completely useless. The **majority-class accuracy** tells us how much a model must achieve to be of any interest. For MNIST it is low, since the dataset is well balanced:

In [ ]:
print('Accuracy of always predicting the most frequent class:', np.max(counts)/np.sum(counts))

#### Visualisation per Class

Next, we are going to look at random different samples from the same class. The objective is to get a feeling about the amount of variability we can expect from samples corresponding to the same class (the so called intra-class variance). The differences between samples of *different* classes are called the inter-class variance.

In [ ]:
# Extract the samples with the given class ID
class_id = 5
x_class = x_train[y_train == class_id]

# Plot random samples with class ID
for ii in range(15):
    idx = np.random.randint(0, len(x_class))
    plt.subplot(3,5,ii+1), plt.imshow(x_class[idx, ...], cmap='gray'), plt.axis(False)

### Average Number Templates

We can observe that the samples do exhibit a certain amount of intra-class variance. However, can we say how a certain digit (out of the 10 possible) looks like on average? This is what we are going to do next, i.e., to plot the "average digit" for each class.

In [ ]:
for class_id in range(num_classes):
    # Average over all the samples of the current class
    x_class = x_train[y_train == class_id]/255
    avg = np.mean(x_class, axis=0)

    # Show the result
    plt.subplot(2,5,class_id+1), plt.imshow(avg, cmap='gray'), plt.title(class_id)

Furthermore, one can also look at the (relative) average area that each digit occupies on the image, i.e., the fraction of non-zero pixels. From the numbers below, conclude that the digit 1 is by far the smallest one (as expected) meanwhile the digits 0 (somehow surprisingly) and 8 (as expected) are the largest ones.

In [ ]:
areas = []
for class_id in range(num_classes):
    x_class = x_train[y_train == class_id]/255
    areas.append(np.mean(x_class > 0))
    print('Avg rel area for', class_id, ':', areas[-1])

plt.bar(range(num_classes), areas), plt.grid(True)
plt.xticks(range(num_classes))
plt.xlabel('Class ID'), plt.ylabel('average relative area')